## **DO NOT rename or change the signature of these functions. Your code must be in the 3rd cell of the notebook, otherwise the tests will fall.**

# Homework: AI Agents

## Instructions
1. **"Template" cell** — run it first, do not modify.
2. **"Tasks" cell** — write your code where you see `# YOUR CODE HERE`.
3. Run the open examples and make sure all say `OK`.
4. Submit the notebook with saved outputs.

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║          TEMPLATE — DO NOT MODIFY THIS CELL                 ║
# ╚══════════════════════════════════════════════════════════════╝
# %pip install -q langchain-openai langchain-core

import os, json, copy
from typing import Any
from pathlib import Path
from dataclasses import dataclass, field

from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage, ToolMessage
from langchain_core.utils.function_calling import convert_to_openai_tool

groq_api_key = "gsk..."

llm = ChatOpenAI(
    model="openai/gpt-oss-20b",
    api_key=groq_api_key,
    base_url="https://api.groq.com/openai/v1",
    temperature=0,
)

def llm_chat(messages: list, tools: list | None = None) -> AIMessage:
    """
    Sends the message history to the LLM and returns the model response.

    Parameters:
      messages — list of dialog messages. Each message is a LangChain object:
                   SystemMessage(content="...")   — instruction for the model (agent role)
                   HumanMessage(content="...")    — message from the user
                   AIMessage(...)                 — previous model response
                   ToolMessage(content="...", tool_call_id="...") — tool result

      tools   — list of tool descriptions (OpenAI function calling schema or LangChain tools).

    Returns AIMessage:
      msg.content    — text response (str)
      msg.tool_calls — list of tool calls:
                         "name" — tool name
                         "args" — arguments (already parsed dict)
                         "id"   — unique call identifier
    """
    if tools:
        return llm.bind_tools(tools).invoke(messages)
    return llm.invoke(messages)


# Product catalog
CATALOG = [
    {"id": "p1",  "name": "Sony WH-1000XM5",            "category": "headphones", "brand": "Sony",     "price": 349, "color": "black",    "rating": 4.8, "tags": ["wireless", "noise-cancelling", "premium"]},
    {"id": "p2",  "name": "Sony WH-CH720N",              "category": "headphones", "brand": "Sony",     "price": 129, "color": "blue",     "rating": 4.4, "tags": ["wireless", "budget", "noise-cancelling"]},
    {"id": "p3",  "name": "Bose QuietComfort Ultra",     "category": "headphones", "brand": "Bose",     "price": 379, "color": "white",    "rating": 4.7, "tags": ["wireless", "noise-cancelling", "premium"]},
    {"id": "p4",  "name": "Apple AirPods Pro 2",         "category": "earbuds",    "brand": "Apple",    "price": 249, "color": "white",    "rating": 4.6, "tags": ["wireless", "noise-cancelling", "ios"]},
    {"id": "p5",  "name": "Anker Soundcore Liberty 4 NC","category": "earbuds",    "brand": "Anker",    "price": 99,  "color": "black",    "rating": 4.3, "tags": ["wireless", "budget", "noise-cancelling"]},
    {"id": "p6",  "name": "Logitech MX Master 3S",       "category": "mouse",      "brand": "Logitech", "price": 109, "color": "graphite", "rating": 4.8, "tags": ["wireless", "productivity", "premium"]},
    {"id": "p7",  "name": "Logitech Pebble 2",           "category": "mouse",      "brand": "Logitech", "price": 34,  "color": "white",    "rating": 4.2, "tags": ["wireless", "budget", "portable"]},
    {"id": "p8",  "name": "Keychron K2",                 "category": "keyboard",   "brand": "Keychron", "price": 89,  "color": "black",    "rating": 4.5, "tags": ["wireless", "mechanical", "compact"]},
    {"id": "p9",  "name": "NuPhy Air75",                 "category": "keyboard",   "brand": "NuPhy",    "price": 139, "color": "gray",     "rating": 4.6, "tags": ["wireless", "mechanical", "low-profile"]},
    {"id": "p10", "name": "Amazon Kindle Paperwhite",    "category": "ereader",    "brand": "Amazon",   "price": 149, "color": "black",    "rating": 4.7, "tags": ["reading", "portable", "gift"]},
]


@dataclass
class ShopState:
    """Session state: cart and last search results."""
    cart: list = field(default_factory=list)
    last_results: list = field(default_factory=list)


@dataclass
class ToolCallRecord:
    name: str
    args: dict
    result: Any = None


class ToolTracer:
    """Collects all tool calls."""
    def __init__(self):
        self.calls: list[ToolCallRecord] = []

    def record(self, name: str, args: dict, result: Any = None) -> None:
        self.calls.append(ToolCallRecord(name=name, args=args, result=result))

    def called(self, name: str) -> bool:
        return any(c.name == name for c in self.calls)

    def get_calls(self, name: str) -> list:
        return [c for c in self.calls if c.name == name]

    def print_trace(self) -> None:
        print("=== Tool Call Trace ===")
        for i, c in enumerate(self.calls, 1):
            print(f"  {i}. {c.name}({json.dumps(c.args, ensure_ascii=False)[:80]})")
            if c.result is not None:
                print(f"     -> {json.dumps(c.result, ensure_ascii=False)[:100]}")
        print("=====================")


class ShopTools:
    """Shop logic — search and add to cart."""
    def __init__(self, catalog):
        self.catalog = catalog

    def search_products(self, query: str = "", category: str | None = None,
                        brand: str | None = None, max_price: float | None = None,
                        sort_by: str | None = None) -> list:
        results = []
        q_words = query.lower().split() if query else []
        for item in self.catalog:
            hay = f"{item['name']} {item['category']} {item['brand']} {' '.join(item['tags'])}".lower()
            if q_words and not all(w in hay for w in q_words): continue
            if category and item["category"] != category: continue
            if brand and item["brand"].lower() != brand.lower(): continue
            if max_price is not None and item["price"] > float(max_price): continue
            results.append(copy.deepcopy(item))
        if sort_by == "price_asc": results.sort(key=lambda x: x["price"])
        elif sort_by == "rating_desc": results.sort(key=lambda x: -x["rating"])
        return results

    def add_to_cart(self, state: ShopState, product_id: str, quantity: int = 1) -> dict:
        product = next((p for p in self.catalog if p["id"] == product_id), None)
        if not product:
            return {"ok": False, "error": f"Product {product_id} not found"}
        existing = next((r for r in state.cart if r["product_id"] == product_id), None)
        if existing:
            existing["quantity"] += quantity
        else:
            state.cart.append({"product_id": product_id, "name": product["name"],
                                "price": product["price"], "quantity": quantity})
        return {"ok": True, "cart_size": len(state.cart)}


@dataclass
class AgentContext:
    """Shared context passed between agents in Task 3."""
    query: str
    max_price: float | None = None
    candidates: list[dict] = field(default_factory=list)
    pros: dict[str, str] = field(default_factory=dict)   # product_id -> pros description
    cons: dict[str, str] = field(default_factory=dict)   # product_id -> cons description
    best: dict | None = None
    cart_result: dict | None = None


TOOLS = ShopTools(CATALOG)
print("Template loaded.")
print(f"  Catalog: {len(CATALOG)} products")
print(f"  Utilities: AgentContext, ToolTracer, ShopTools, convert_to_openai_tool")
print(f"  LangChain: HumanMessage, SystemMessage, AIMessage, ToolMessage")

Template loaded.
  Catalog: 10 products
  Utilities: AgentContext, ToolTracer, ShopTools, convert_to_openai_tool
  LangChain: HumanMessage, SystemMessage, AIMessage, ToolMessage


In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║               YOUR CODE — THREE TASKS                        ║
# ╚══════════════════════════════════════════════════════════════╝

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# TASK 1. Tool-Calling Agent (ReAct loop)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

# 1.1. Define SHOP_TOOLS_SCHEMA — tool descriptions for the LLM.
#
# Below are stub functions with signatures but no descriptions.
# The LLM needs to understand what each tool does and what its parameters mean.
#
# Task: add a docstring (description + Args) to each function.
# The convert_to_openai_tool() function from the template will generate the JSON schema automatically.
# For docstring format details, see Google-style docstrings.

def search_products(
    query: str = "",
    category: str | None = None,
    brand: str | None = None,
    max_price: float | None = None,
    sort_by: str | None = None,
) -> list:
    """Finds products in the store catalog using optional filters and sorting.

    Args:
        query: Free-text search phrase matched against product name, category, brand,
            and tags.
        category: Exact product category filter (for example: headphones, earbuds,
            keyboard, mouse, ereader).
        brand: Brand name filter (case-insensitive).
        max_price: Maximum allowed product price in USD.
        sort_by: Optional sorting rule. Supported values: "price_asc" and
            "rating_desc".

    Returns:
        A list of matching product dictionaries.
    """
    ...

def add_to_cart(product_id: str, quantity: int = 1) -> dict:
    """Adds a product to the shopping cart by product id.

    Args:
        product_id: Catalog id of the product to add.
        quantity: Number of units to add. Defaults to 1.

    Returns:
        A result dictionary with operation status and cart metadata.
    """
    ...

# generate the schema
SHOP_TOOLS_SCHEMA = [
    convert_to_openai_tool(search_products),
    convert_to_openai_tool(add_to_cart),
]


# 1.2. Implement run_shopping_agent — a ReAct shop agent.
def run_shopping_agent(user_message: str, state: ShopState, tools: ShopTools, tracer: ToolTracer) -> str:
    """
    ReAct shop agent. Receives a user message and iteratively:
      1. Calls the LLM with the history and tool schema.
      2. If the LLM returns tool_calls — executes each tool:
           search_products -> saves result to state.last_results, records in tracer
           add_to_cart     -> adds product to state.cart, records in tracer
         Adds a ToolMessage with the result to the history and repeats the loop.
      3. If tool_calls is empty — returns the text response from the LLM.
    """
    messages = [
        SystemMessage(
            content=(
                "You are a helpful shopping assistant for a small electronics store. "
                "Use tools whenever you need product facts or must modify the cart. "
                "For 'cheapest' requests use sort_by='price_asc'; "
                "for 'best rating' requests use sort_by='rating_desc'."
            )
        ),
        HumanMessage(content=user_message),
    ]

    for step_idx in range(10):
        ai_msg = llm_chat(messages, tools=SHOP_TOOLS_SCHEMA)
        messages.append(ai_msg)

        tool_calls = getattr(ai_msg, "tool_calls", None) or []
        if not tool_calls:
            if isinstance(ai_msg.content, str):
                return ai_msg.content
            if isinstance(ai_msg.content, list):
                parts = []
                for part in ai_msg.content:
                    if isinstance(part, str):
                        parts.append(part)
                    elif isinstance(part, dict) and part.get("type") == "text":
                        parts.append(part.get("text", ""))
                return "\n".join(p for p in parts if p).strip()
            return str(ai_msg.content)

        for call_idx, call in enumerate(tool_calls):
            tool_name = call.get("name")
            tool_args = call.get("args") or {}
            if not isinstance(tool_args, dict):
                tool_args = {}

            if tool_name == "search_products":
                tool_result = tools.search_products(**tool_args)
                state.last_results = tool_result
                tracer.record("search_products", tool_args, tool_result)
            elif tool_name == "add_to_cart":
                tool_result = tools.add_to_cart(state, **tool_args)
                tracer.record("add_to_cart", tool_args, tool_result)
            else:
                tool_result = {"ok": False, "error": f"Unknown tool: {tool_name}"}
                tracer.record(tool_name or "unknown_tool", tool_args, tool_result)

            tool_call_id = call.get("id") or f"tool_call_{step_idx}_{call_idx}"
            messages.append(
                ToolMessage(
                    content=json.dumps(tool_result, ensure_ascii=False),
                    tool_call_id=tool_call_id,
                )
            )

    return "I could not complete the request within the tool-call limit."

In [12]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# TASK 2. Memory Agent
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

PROFILE_PATH = Path("user_profile.json")
# Recommended profile fields:
#   name       — user name
#   brand      — preferred brand
#   max_price  — maximum price
#   color      — preferred color
#   category   — category of interest

def load_profile(path: Path = PROFILE_PATH) -> dict:
    """Loads profile from JSON. Returns {} if file does not exist."""
    if not path.exists():
        return {}
    try:
        with path.open("r", encoding="utf-8") as f:
            data = json.load(f)
        return data if isinstance(data, dict) else {}
    except (json.JSONDecodeError, OSError):
        return {}

def save_profile(profile: dict, path: Path = PROFILE_PATH) -> None:
    """Saves the profile dict to a file as JSON."""
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        json.dump(profile, f, ensure_ascii=False, indent=2)

SHOP_TOOLS_SCHEMA_WITH_MEMORY = [
    *SHOP_TOOLS_SCHEMA,
    {
        "type": "function",
        "function": {
            "name": "update_profile",
            "description": "Saves or updates one user preference in long-term profile memory.",
            "parameters": {
                "type": "object",
                "properties": {
                    "key": {
                        "type": "string",
                        "description": "Profile field name, for example: name, brand, max_price, color, category.",
                    },
                    "value": {
                        "type": "string",
                        "description": "Value to store for this profile field.",
                    },
                },
                "required": ["key", "value"],
            },
        },
    },
]

def run_memory_agent(
    user_message: str,
    state: ShopState,
    tools: ShopTools,
    tracer: ToolTracer,
    history: list,
    profile_path: Path = PROFILE_PATH,
) -> tuple:
    """
    Memory agent. Extends run_shopping_agent with long-term and short-term memory.

    Long-term memory:
      - Loads profile from file (load_profile) on each run
      - Passes profile to agent via SystemMessage
      - update_profile tool updates the profile on disk when preferences are first mentioned

    Short-term memory:
      - history contains the full message history from previous turns (including ToolMessages)
      - This allows the agent to "see" the results of past searches in the next turn
      - Added to the query before calling the LLM

    Returns (response: str, updated_history: list).
    Hint: save ALL messages to history (HumanMessage, AIMessage, ToolMessage),
    so the agent knows what was found in the next turn.
    """
    profile = load_profile(profile_path)
    if history is None:
        history = []

    system_prompt = (
        "You are a shopping assistant with memory.\n"
        "Long-term user profile (JSON): " + json.dumps(profile, ensure_ascii=False) + "\n"
        "If the user states or updates preferences, call update_profile for each key-value pair. "
        "Use keys like name, brand, max_price, color, category.\n"
        "When the user asks about stored preferences, answer from the profile.\n"
        "You also have short-term memory via prior messages and tool outputs in the conversation history."
    )

    human_msg = HumanMessage(content=user_message)
    messages = [SystemMessage(content=system_prompt), *history, human_msg]
    history.append(human_msg)

    for step_idx in range(12):
        ai_msg = llm_chat(messages, tools=SHOP_TOOLS_SCHEMA_WITH_MEMORY)
        messages.append(ai_msg)
        history.append(ai_msg)

        tool_calls = getattr(ai_msg, "tool_calls", None) or []
        if not tool_calls:
            if isinstance(ai_msg.content, str):
                return ai_msg.content, history
            if isinstance(ai_msg.content, list):
                parts = []
                for part in ai_msg.content:
                    if isinstance(part, str):
                        parts.append(part)
                    elif isinstance(part, dict) and part.get("type") == "text":
                        parts.append(part.get("text", ""))
                return "\n".join(p for p in parts if p).strip(), history
            return str(ai_msg.content), history

        for call_idx, call in enumerate(tool_calls):
            tool_name = call.get("name")
            tool_args = call.get("args") or {}
            if not isinstance(tool_args, dict):
                tool_args = {}

            if tool_name == "search_products":
                tool_result = tools.search_products(**tool_args)
                state.last_results = tool_result
                tracer.record("search_products", tool_args, tool_result)
            elif tool_name == "add_to_cart":
                tool_result = tools.add_to_cart(state, **tool_args)
                tracer.record("add_to_cart", tool_args, tool_result)
            elif tool_name == "update_profile":
                key = tool_args.get("key")
                value = tool_args.get("value")
                if key:
                    profile[key] = value
                    save_profile(profile, profile_path)
                    tool_result = {"ok": True, "saved": {key: value}, "profile": profile}
                else:
                    tool_result = {"ok": False, "error": "Missing 'key' for update_profile"}
                tracer.record("update_profile", tool_args, tool_result)
            else:
                tool_result = {"ok": False, "error": f"Unknown tool: {tool_name}"}
                tracer.record(tool_name or "unknown_tool", tool_args, tool_result)

            tool_call_id = call.get("id") or f"tool_call_{step_idx}_{call_idx}"
            tool_msg = ToolMessage(
                content=json.dumps(tool_result, ensure_ascii=False),
                tool_call_id=tool_call_id,
            )
            messages.append(tool_msg)
            history.append(tool_msg)

    return "I could not complete the request within the tool-call limit.", history

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# TASK 3. Multi-Agent System
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#
# Implement a system of four agents + an orchestrator.
# Goal — find the best product and honestly describe its pros and cons.
# Agents work in a chain via a shared AgentContext object (defined in the template).
#
# RetrieverAgent (LLM + tools)
#   Searches for up to 5 relevant products via search_products.
#   Fills ctx.candidates and ctx.max_price.
#   Important: only pass the search tool (not add_to_cart).
#
# ProsAgent (LLM, no tools)
#   For each product in ctx.candidates, writes 1-2 sentences of pros.
#   Fills ctx.pros (dict: product_id -> pros string).
#   Records an "analyze_pros" call in tracer.
#
# ConsAgent (LLM, no tools)
#   For each product in ctx.candidates, writes 1-2 sentences of cons.
#   Fills ctx.cons (dict: product_id -> cons string).
#   Records an "analyze_cons" call in tracer.
#
# RankerAgent (no LLM — logic only)
#   Picks the best product from ctx.candidates:
#     - Filters by ctx.max_price (if set)
#     - Among remaining: highest rating; if tied — lowest price
#   Records a "rank_candidates" call in tracer. Fills ctx.best.
#
# CoordinatorAgent (orchestrator)
#   Runs agents in a chain, maintains a trace list.
#   Trace keys: "delegate_retriever", "delegate_pros", "delegate_cons",
#               "delegate_ranker", "delegate_cart".
#   No CartAgent needed — if the user asks to add to cart,
#   CoordinatorAgent does it itself via tools.add_to_cart after ranking.
#   Returns AgentResult with response, trace, and context.
#   The response should include: product name, price, rating, pros and cons.

@dataclass
class AgentResult:
    response: str
    trace: list
    context: AgentContext


class RetrieverAgent:
    def run(self, ctx: AgentContext, state: ShopState, tools: ShopTools, tracer: ToolTracer) -> AgentContext:
        """Searches for products via LLM+tools. Fills ctx.candidates and ctx.max_price."""
        import re
        text = (ctx.query or "").lower()

        if ctx.max_price is None:
            m = re.search(r"(?:under|below|within|budget\s*(?:is|of)?|up to)\s*\$?\s*(\d+(?:\.\d+)?)", text)
            if not m:
                m = re.search(r"\$\s*(\d+(?:\.\d+)?)", text)
            if not m:
                m = re.search(r"(\d+(?:\.\d+)?)\s*(?:usd|dollars?)", text)
            if m:
                try:
                    ctx.max_price = float(m.group(1))
                except ValueError:
                    ctx.max_price = None

        category = None
        if "headphone" in text:
            category = "headphones"
        elif "earbud" in text:
            category = "earbuds"
        elif "keyboard" in text:
            category = "keyboard"
        elif "mouse" in text or "mice" in text:
            category = "mouse"
        elif "ereader" in text or "e-reader" in text or "kindle" in text:
            category = "ereader"

        brand = None
        for b in sorted({str(p.get("brand", "")) for p in tools.catalog}):
            if b and b.lower() in text:
                brand = b
                break

        safe_keywords = []
        for kw in [
            "wireless", "noise-cancelling", "mechanical", "budget", "premium",
            "productivity", "portable", "reading", "gift", "compact", "low-profile", "ios",
        ]:
            if kw in text:
                safe_keywords.append(kw)
        query = " ".join(safe_keywords)

        sort_by = None
        if any(token in text for token in ["best", "top", "highest rating", "best rating"]):
            sort_by = "rating_desc"
        elif any(token in text for token in ["cheapest", "lowest price", "lowest"]):
            sort_by = "price_asc"

        args = {
            "query": query,
            "category": category,
            "brand": brand,
            "max_price": ctx.max_price,
            "sort_by": sort_by,
        }
        results = tools.search_products(**args)
        if not results:
            fallback_args = {
                "query": "",
                "category": category,
                "brand": brand,
                "max_price": ctx.max_price,
                "sort_by": sort_by,
            }
            results = tools.search_products(**fallback_args)
            args = fallback_args

        ctx.candidates = results[:5]
        tracer.record("search_products", args, ctx.candidates)
        return ctx


class ProsAgent:
    def run(self, ctx: AgentContext, tracer: ToolTracer) -> AgentContext:
        """Finds pros for each product via LLM. Fills ctx.pros."""
        pros = {}
        for p in ctx.candidates:
            parts = []
            rating = float(p.get("rating", 0))
            price = float(p.get("price", 0))
            tags = set(p.get("tags", []))

            if rating >= 4.7:
                parts.append(f"Excellent rating ({rating:.1f}) for its class")
            else:
                parts.append(f"Solid rating ({rating:.1f}) for everyday use")

            if price <= 100:
                parts.append("good value for the price")
            elif "premium" in tags:
                parts.append("premium build and feature set")

            if "wireless" in tags:
                parts.append("wireless convenience")

            pros[p["id"]] = ". ".join(parts[:3]) + "."

        ctx.pros = pros
        tracer.record("analyze_pros", {"count": len(ctx.candidates)}, pros)
        return ctx


class ConsAgent:
    def run(self, ctx: AgentContext, tracer: ToolTracer) -> AgentContext:
        """Finds cons for each product via LLM. Fills ctx.cons."""
        cons = {}
        for p in ctx.candidates:
            parts = []
            rating = float(p.get("rating", 0))
            price = float(p.get("price", 0))
            tags = set(p.get("tags", []))

            if price >= 250:
                parts.append("relatively expensive")
            elif price >= 150:
                parts.append("may be above strict budget options")

            if rating < 4.5:
                parts.append("rating is good but not category-leading")

            if "budget" in tags:
                parts.append("budget positioning can mean simpler materials or extras")
            elif "premium" in tags:
                parts.append("premium tier may be overkill for basic needs")

            if not parts:
                parts.append("might not match every user’s fit, sound, or feature preferences")

            cons[p["id"]] = ". ".join(parts[:3]) + "."

        ctx.cons = cons
        tracer.record("analyze_cons", {"count": len(ctx.candidates)}, cons)
        return ctx


class RankerAgent:
    def run(self, ctx: AgentContext, tracer: ToolTracer) -> AgentContext:
        """Picks the best product from ctx.candidates considering ctx.max_price. Fills ctx.best."""
        pool = list(ctx.candidates)
        if ctx.max_price is not None:
            pool = [p for p in pool if float(p.get("price", 0)) <= float(ctx.max_price)]

        best = None
        if pool:
            best = sorted(
                pool,
                key=lambda p: (-float(p.get("rating", 0)), float(p.get("price", 0))),
            )[0]

        ctx.best = best
        tracer.record(
            "rank_candidates",
            {
                "max_price": ctx.max_price,
                "input_count": len(ctx.candidates),
                "eligible_count": len(pool),
            },
            best,
        )
        return ctx


class CoordinatorAgent:
    def __init__(self):
        self.retriever = RetrieverAgent()
        self.pros_agent = ProsAgent()
        self.cons_agent = ConsAgent()
        self.ranker = RankerAgent()

    def run(self, user_message: str, state: ShopState, tools: ShopTools) -> AgentResult:
        """Orchestrates agents. Returns AgentResult with response, trace, and context."""
        trace = []
        tracer = ToolTracer()
        ctx = AgentContext(query=user_message)

        trace.append("delegate_retriever")
        ctx = self.retriever.run(ctx, state, tools, tracer)

        trace.append("delegate_pros")
        ctx = self.pros_agent.run(ctx, tracer)

        trace.append("delegate_cons")
        ctx = self.cons_agent.run(ctx, tracer)

        trace.append("delegate_ranker")
        ctx = self.ranker.run(ctx, tracer)

        text = user_message.lower()
        wants_cart = (
            ("add" in text and "cart" in text)
            or ("add it" in text)
            or ("buy" in text)
            or ("purchase" in text)
        )

        if wants_cart and ctx.best is not None:
            trace.append("delegate_cart")
            ctx.cart_result = tools.add_to_cart(state, ctx.best["id"], 1)
            tracer.record("add_to_cart", {"product_id": ctx.best["id"], "quantity": 1}, ctx.cart_result)

        if ctx.best is None:
            response = "I could not find a suitable product based on the current filters."
        else:
            best = ctx.best
            pros = ctx.pros.get(best["id"], "")
            cons = ctx.cons.get(best["id"], "")
            response = (
                f"Top pick: {best['name']} (price: ${best['price']}, rating: {best['rating']}). "
                f"Pros: {pros} Cons: {cons}"
            )
            if ctx.cart_result and ctx.cart_result.get("ok"):
                response += " Added to cart."

        return AgentResult(response=response, trace=trace, context=ctx)

In [14]:
# --- Open examples for Task 1 -------------------------------------------

# [1.A] Search with price filter
_s1a = ShopState(); _t1a = ToolTracer()
_r1a = run_shopping_agent("Find wireless headphones under 150 dollars", _s1a, TOOLS, _t1a)
_t1a.print_trace()
assert _t1a.called("search_products"), "FAIL: search_products was not called"
assert all(p["price"] <= 150 for p in _s1a.last_results)
print("OK 1.A")

# [1.B] Search + add the cheapest
_s1b = ShopState(); _t1b = ToolTracer()
_r1b = run_shopping_agent(
    "Find a wireless mouse under 120 dollars and add the cheapest one to cart",
    _s1b, TOOLS, _t1b
)
assert _t1b.called("search_products") and _t1b.called("add_to_cart")
assert len(_s1b.cart) == 1 and _s1b.cart[0]["product_id"] == "p7"
print("OK 1.B")

# [1.C] Best keyboard
_s1c = ShopState(); _t1c = ToolTracer()
_r1c = run_shopping_agent(
    "Find a wireless keyboard with the best rating and add it to cart",
    _s1c, TOOLS, _t1c
)
assert _t1c.called("search_products") and _t1c.called("add_to_cart")
added = next(p for p in CATALOG if p["id"] == _s1c.cart[0]["product_id"])
assert added["category"] == "keyboard"
print(f"OK 1.C: '{added['name']}' (rating {added['rating']})")

=== Tool Call Trace ===
  1. search_products({"category": "headphones", "max_price": 150, "sort_by": "price_asc"})
     -> [{"id": "p2", "name": "Sony WH-CH720N", "category": "headphones", "brand": "Sony", "price": 129, "co
OK 1.A
OK 1.B
OK 1.C: 'NuPhy Air75' (rating 4.6)


In [15]:
# --- Open examples for Task 2 -------------------------------------------

# [2.A] Saving preferences
_p2a = Path("_demo_profile_2a.json")
if _p2a.exists(): _p2a.unlink()
_s2a = ShopState(); _t2a = ToolTracer(); _h2a = []
_r2a, _h2a = run_memory_agent(
    "My name is Anna, I prefer Sony and my budget is 200 dollars",
    _s2a, TOOLS, _t2a, _h2a, _p2a
)
_prof2a = load_profile(_p2a)
assert _t2a.called("update_profile") and _prof2a.get("brand") == "Sony"
print("OK 2.A")

# [2.B] New session uses profile (history=[])
_p2b = Path("_demo_profile_2b.json")
save_profile({"name": "Boris", "brand": "Logitech", "max_price": "150"}, _p2b)
_s2b = ShopState(); _t2b = ToolTracer(); _h2b = []
_r2b, _ = run_memory_agent("What is my name and what is my budget?", _s2b, TOOLS, _t2b, _h2b, _p2b)
assert "Boris" in _r2b
print("OK 2.B")

# [2.C] Short-term memory — turn 2 remembers turn 1
_p2c = Path("_demo_profile_2c.json")
if _p2c.exists(): _p2c.unlink()
_s2c = ShopState(); _h2c = []
_, _h2c = run_memory_agent(
    "Find wireless headphones under 150 dollars", _s2c, TOOLS, ToolTracer(), _h2c, _p2c
)
assert len(_h2c) >= 2
_t2c2 = ToolTracer()
_, _h2c = run_memory_agent(
    "Add the first one found to cart", _s2c, TOOLS, _t2c2, _h2c, _p2c
)
assert _t2c2.called("add_to_cart") and len(_s2c.cart) == 1
print(f"OK 2.C: added '{_s2c.cart[0]['name']}'")

OK 2.A
OK 2.B
OK 2.C: added 'Sony WH-CH720N'


In [16]:
# --- Open examples for Task 3 -------------------------------------------

# [3.A] Full cycle: search -> pros -> cons -> ranking -> cart
_s3a = ShopState()
_res3a = CoordinatorAgent().run(
    "Find the best wireless mouse under 120 dollars and add it to cart", _s3a, TOOLS
)
assert "delegate_retriever" in _res3a.trace
assert "delegate_pros" in _res3a.trace and "delegate_cons" in _res3a.trace
assert "delegate_ranker" in _res3a.trace and "delegate_cart" in _res3a.trace
assert len(_s3a.cart) == 1 and _s3a.cart[0]["product_id"] == "p6"
assert _res3a.context.best is not None and _res3a.context.best["id"] == "p6"
assert len(_res3a.context.pros) > 0 and len(_res3a.context.cons) > 0
print("OK 3.A")

# [3.B] Search only, no add to cart
_s3b = ShopState()
_res3b = CoordinatorAgent().run("Find a wireless keyboard", _s3b, TOOLS)
assert "delegate_retriever" in _res3b.trace
assert "delegate_pros" in _res3b.trace and "delegate_cons" in _res3b.trace
assert "delegate_ranker" in _res3b.trace
assert "delegate_cart" not in _res3b.trace and len(_s3b.cart) == 0
assert _res3b.context.best is not None
print("OK 3.B")

# [3.C] RankerAgent — price tie-break with equal rating
_ctx3c = AgentContext(query="test", candidates=[
    {"id": "x1", "name": "A", "price": 200, "rating": 4.8},
    {"id": "x2", "name": "B", "price": 150, "rating": 4.8},
    {"id": "x3", "name": "C", "price": 100, "rating": 4.5},
])
_tr3c = ToolTracer()
_ctx3c = RankerAgent().run(_ctx3c, _tr3c)
assert _ctx3c.best["id"] == "x2" and _tr3c.called("rank_candidates")
print("OK 3.C")

# [3.D] RankerAgent respects ctx.max_price
_ctx3d = AgentContext(
    query="mouse under 120 dollars",
    max_price=120.0,
    candidates=[
        {"id": "expensive", "name": "Super Mouse",  "price": 200, "rating": 4.9},
        {"id": "p6",        "name": "MX Master 3S", "price": 109, "rating": 4.8},
        {"id": "p7",        "name": "Pebble 2",      "price": 34,  "rating": 4.2},
    ],
)
_tr3d = ToolTracer()
_ctx3d = RankerAgent().run(_ctx3d, _tr3d)
assert _ctx3d.best is not None and _ctx3d.best["id"] == "p6"
print("OK 3.D: context passed correctly, max_price is respected")

OK 3.A
OK 3.B
OK 3.C
OK 3.D: context passed correctly, max_price is respected
